In [ ]:
# Cell 1 - Imports and configuration

import os
import pickle

import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()



# Configuration

INPUT_FILE = os.getenv(
    "EMBEDDINGS_INPUT_FILE_PATH",
    "./data/questions_answers.csv"
)

EMBEDDINGS_OUTPUT_FILE = os.getenv(
    "EMBEDDINGS_OUTPUT_FILE_PATH",
    "./data/embedded_questions.pkl"
)

EMBEDDING_MODEL = os.getenv(
    "EMBEDDING_MODEL",
    "BAAI/bge-large-en-v1.5"
)

print(f"Input file: {INPUT_FILE}")
print(f"Output file: {EMBEDDINGS_OUTPUT_FILE}")
print(f"Embedding model: {EMBEDDING_MODEL}")

In [ ]:
# Cell 2 - Load dataset

df = pd.read_csv(INPUT_FILE)

print(f"Number of records: {len(df)}")
print(df.head())

In [ ]:
# Cell 3 - Validate dataset

required_columns = [
    "question",
    "answer"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

df = df.dropna(
    subset=["question", "answer"]
).reset_index(drop=True)

print(f"Valid records: {len(df)}")

In [ ]:
# Cell 4 - Create documents to embed

df["document"] = (
    "Question: "
    + df["question"].astype(str)
    + "\n\nAnswer: "
    + df["answer"].astype(str)
)

print(df["document"].iloc[0])

In [ ]:
# Cell 5 - Load embedding model

print("Loading embedding model...")

model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Embedding model loaded.")

In [ ]:
# Cell 6 - Generate embeddings

documents = df["document"].tolist()

print(
    f"Generating embeddings for {len(documents)} documents..."
)

embeddings = model.encode(
    documents,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(
    f"Embedding shape: {embeddings.shape}"
)

In [ ]:
# Cell 7 - Create IDs

ids = [
    f"question_{i:06d}"
    for i in range(len(df))
]

print(ids[:5])

In [ ]:
# Cell 8 - Prepare metadata

metadata = [
    {
        "question": str(row["question"]),
        "answer": str(row["answer"]),
    }
    for _, row in df.iterrows()
]

print(metadata[0])

In [ ]:
# Cell 9 - Create final embedded dataset

embedded_data = {
    "ids": ids,
    "documents": documents,
    "metadatas": metadata,
    "embeddings": embeddings,
}

print(
    f"Prepared {len(ids)} embedded records."
)

In [ ]:
# Cell 10 - Save embeddings

output_directory = os.path.dirname(
    EMBEDDINGS_OUTPUT_FILE
)

if output_directory:
    os.makedirs(
        output_directory,
        exist_ok=True
    )

with open(
    EMBEDDINGS_OUTPUT_FILE,
    "wb"
) as file:

    pickle.dump(
        embedded_data,
        file
    )

print(
    f"Saved embedded dataset to: "
    f"{EMBEDDINGS_OUTPUT_FILE}"
)

In [ ]:
# Cell 11 - Verify saved data

with open(
    EMBEDDINGS_OUTPUT_FILE,
    "rb"
) as file:

    loaded_data = pickle.load(file)

print(
    f"IDs: {len(loaded_data['ids'])}"
)

print(
    f"Documents: {len(loaded_data['documents'])}"
)

print(
    f"Metadata: {len(loaded_data['metadatas'])}"
)

print(
    f"Embeddings shape: "
    f"{loaded_data['embeddings'].shape}"
)

print("\nFirst record:")
print("ID:", loaded_data["ids"][0])
print("Document:", loaded_data["documents"][0])
print("Metadata:", loaded_data["metadatas"][0])